# Reproduce LGEA — Table 2 & Table 3 (FBDB15K / FBYG15K)

*Breaking the Noise Barrier* · [paper](https://aclanthology.org/2025.emnlp-main.1684.pdf) ·
[code](https://github.com/alusang/LGEA-framework)

**Table 2 — main results (20% seeds)**

| | Hits@1 | Hits@10 | MRR |
|---|---|---|---|
| FBDB15K | .801 | .910 | .842 |
| FBYG15K | .850 | .942 | .883 |

**Table 3 — ablations** (FBDB15K / FBYG15K, Hits@1)

| row | FBDB15K | FBYG15K |
|---|---|---|
| LGEA | .801 | .850 |
| w/o img | .740 | .767 |
| w/o rel | .737 | .813 |
| w/o structure | .440 | .678 |
| w/o att | .655 | .563 |
| w/o llm | .682 | .558 |
| **w/o mask** | **.714** | **.818** |
| w/o it | .684 | .785 |

---

## Read this before running — two things change how you read your own numbers

### 1. Your runs have no visual mask, so your "LGEA" row *is* the paper's `w/o mask` row

The mask needs BLIP over the 50GB Baidu-only image dump. `model/NewEA.py:269` turns
`mask_path=None` into an all-ones mask. So **compare your LGEA against `.714/.868/.772`
(FBDB15K) and `.818/.917/.854` (FBYG15K)** — the bolded row above — not against `.801`/`.850`.

That makes `w/o mask` a free, directly comparable anchor. The other ablation rows in the
paper *do* include the mask, so yours sit one modality lower than theirs; the **shape** of
the ablation (which modality matters most) is what transfers, not the absolute values.

### 2. The paper's headline numbers are ITERATIVE, and the DBP15K notebook did not use that

Section 5.1: *"For each setting, models are trained iteratively, where alignment predictions
from previous rounds are gradually incorporated."* Table 3's `w/o it` row (`.684` vs `.801`
on FBDB15K) is what turning that off costs — **11.7 points**. So `USE_IL = True` here, which
means `--il --il_start 500 --epoch 1000` (`run_newea_il.sh`) instead of 250 epochs.

If you ran the DBP15K notebook before this change, that run was non-iterative, which likely
explains a good part of its `.9578` vs `.970`.

---

| | |
|---|---|
| **Accelerator** | **GPU T4 ×2** |
| **Internet** | ON |
| **Attributes here are purely numeric** | FB15K/DB15K/YAGO15K ship only `NumericalTriples` (longitude, population, gross, runtime). No `(name, ...)` pairs, unlike DBP15K — so no surface-form leakage. This is the cleaner test of the method. |

## 0 · Config

In [1]:
import os

DATASETS = ['FBDB15K']            # + 'FBYG15K'
DATA_RATE = '0.2'                 # paper sec 5.1: 20% seeds for the FB datasets
USE_IL    = True                  # paper sec 5.1; Table 3 'w/o it' shows what False costs

# Which Table 3 rows to produce. 'LGEA' is the Table 2 run and doubles as 'w/o mask'.
# RUNS = ['LGEA']
RUNS = ['LGEA', 'w/o img', 'w/o rel', 'w/o structure', 'w/o att', 'w/o llm', 'w/o it']

BUNDLE = None                     # e.g. '/kaggle/input/lgea-data-fb'

BASE_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'      # gated: needs HF_TOKEN secret
# BASE_MODEL = 'NousResearch/Meta-Llama-3-8B-Instruct'  # ungated mirror, same weights
L2V_MNTP = 'McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp'
L2V_SUP  = 'McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised'
CODE_REV = '1af5f3140c42'         # last revision written against the transformers 4.44 API

BATCH_SIZE = 16
MAX_LENGTH = 512
MAX_ATTR_PER_ENTITY = 40
SEED = '42'

# Reuse the LLM embeddings from a previous session instead of recomputing (~1.4h/dataset).
EMB_INPUT = None       # e.g. '/kaggle/input/lgea-fb-embeddings'
SAVE_MODEL = True      # the repo ships --save_model 0; this writes ~60MB per run

FORCE_DATA = FORCE_EMBED = FORCE_TRAIN = False

TEMP = '/kaggle/temp'
WORK = '/kaggle/working'
ROOT = f'{WORK}/lgea'
CODE = f'{ROOT}/code/LGEA-framework'
DATA = f'{ROOT}/data/mmkg'
PKL_DIR = f'{DATA}/pkls'
MMKB = f'{TEMP}/mmkb'

def ds_dir(d):      return f'{DATA}/{d}/norm'          # config.py forces data_split='norm'
def img_pkl(d):     return f'{d}_id_img_feature_dict.pkl'   # src/data.py:69
def emb_path(s, d): return f'{WORK}/att_sum_{s}_noname_{d}.pkl'
def bag_path(s, d): return f'{WORK}/att_bag_{s}_{d}.pkl'    # the 'w/o llm' features
def done(t):        return os.path.exists(f'{WORK}/.done_{t}')
def mark(t):        open(f'{WORK}/.done_{t}', 'w').close()
def slug(r):        return r.replace('/', '').replace(' ', '_')

for p in (TEMP, ROOT, PKL_DIR, f'{DATA}/dump'):
    os.makedirs(p, exist_ok=True)
for d in DATASETS:
    os.makedirs(ds_dir(d), exist_ok=True)

if not USE_IL and 'w/o it' in RUNS:
    RUNS = [r for r in RUNS if r != 'w/o it']   # identical to LGEA when IL is already off
    print("USE_IL=False, so 'w/o it' would duplicate 'LGEA' -- dropped")

# rough cost, from the measured DBP15K rate (8.0 s/epoch at 38,960 entities)
N_ENT = {'FBDB15K': 27793, 'FBYG15K': 30355}
print(f'{"dataset":9s} {"embed":>7s} {"per run":>9s} {"runs":>5s} {"train":>7s} {"total":>7s}')
grand = 0.0
for d in DATASETS:
    embed_h = 1.5 * N_ENT[d] / 30000
    train_h = sum(8.0 * N_ENT[d] / 38960 * (1000 if (USE_IL and r != 'w/o it') else 250) / 3600
                  for r in RUNS)
    grand += embed_h + train_h
    print(f'{d:9s} {embed_h:6.1f}h {8.0*N_ENT[d]/38960*(1000 if USE_IL else 250)/3600:8.1f}h '
          f'{len(RUNS):5d} {train_h:6.1f}h {embed_h+train_h:6.1f}h')
print(f'\nestimated total: {grand:.1f}h   (Kaggle session limit 12h, quota 30h/week)')
print('IL runs are 1000 epochs vs 250 -- that is where the time goes.')

dataset     embed   per run  runs   train   total
FBDB15K      1.4h      1.6h     7    9.9h   11.3h

estimated total: 11.3h   (Kaggle session limit 12h, quota 30h/week)
IL runs are 1000 epochs vs 250 -- that is where the time goes.


## 1 · Install

Same pins as the DBP15K notebook: `llm2vec` requires `transformers>=4.43.1,<=4.44.2`, and
that is what forces `CODE_REV`.

In [2]:
!pip install -q gdown easydict unidecode 2>&1 | tail -2
!pip install -q "llm2vec==0.2.3" "peft==0.12.0" "transformers==4.44.2" 2>&1 | tail -3

from importlib.metadata import version
for p in ('transformers', 'peft', 'llm2vec', 'torch', 'accelerate'):
    print(f'{p:14s} {version(p)}')
assert version('transformers').startswith('4.44')
!nvidia-smi --query-gpu=name,memory.total --format=csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
transformers   4.44.2
peft           0.12.0
llm2vec        0.2.3
torch          2.10.0+cu128
accelerate     1.13.0
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 2 · Data

Three sources this time:

* **MEAformer package (1.26G)** — `FBDB15K/norm/`, `FBYG15K/norm/` (ent_ids, triples,
  ill_ent_ids, training_attrs) and `pkls/FB*15K_id_img_feature_dict.pkl`.
* **[mmkb](https://github.com/mniepert/mmkb)** — `*_NumericalTriples.txt`, the only place
  with attribute **values**. The MEAformer package ships attribute *names* only.
* the LGEA repo itself.

In [3]:
import subprocess, shutil, glob

if not os.path.exists(CODE):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/alusang/LGEA-framework.git', CODE], check=True)
if not os.path.exists(MMKB):
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/mniepert/mmkb.git', MMKB], check=True)
    print('cloned mmkb')

if BUNDLE:
    src = f'{BUNDLE}/mmkg'
    assert os.path.isdir(src), f'{src} not found; /kaggle/input holds {os.listdir("/kaggle/input")}'
    for d in DATASETS:
        shutil.copytree(f'{src}/{d}/norm', ds_dir(d), dirs_exist_ok=True)
        shutil.copy2(f'{src}/pkls/{img_pkl(d)}', f'{PKL_DIR}/{img_pkl(d)}')
        print(f'{d}: copied from bundle')
elif done('data') and not FORCE_DATA:
    print('data already prepared, skipping')
else:
    !gdown -q 1VIWcc3KDcLcRImeSrF2AyhetBLq_gsnx -O {TEMP}/mmkg.zip
    !mkdir -p {TEMP}/mea && unzip -q -o {TEMP}/mmkg.zip -d {TEMP}/mea

    def find_mmkg(root):
        for dirpath, dirnames, _ in os.walk(root):
            if os.path.basename(dirpath) == 'mmkg' and 'DBP15K' in dirnames:
                return dirpath
        raise SystemExit(f'could not locate an "mmkg" directory under {root}')

    MMKG_SRC = find_mmkg(f'{TEMP}/mea')
    print('mmkg source:', MMKG_SRC)
    for d in DATASETS:
        for n in ['ent_ids_1', 'ent_ids_2', 'ill_ent_ids', 'triples_1', 'triples_2',
                  'training_attrs_1', 'training_attrs_2']:
            shutil.copy2(f'{MMKG_SRC}/{d}/norm/{n}', f'{ds_dir(d)}/{n}')
        shutil.copy2(f'{MMKG_SRC}/pkls/{img_pkl(d)}', f'{PKL_DIR}/{img_pkl(d)}')
        print(f'{d}: copied 7 text files + {img_pkl(d)}')
    mark('data')

cloned mmkb
mmkg source: /kaggle/temp/mea/mmkg
FBDB15K: copied 7 text files + FBDB15K_id_img_feature_dict.pkl


## 3 · Build the attribute files

`FB15K_NumericalTriples.txt` is tab-separated with a bare value; `DB15K`/`YAGO15K` are
N-Triples. Which mmkb KG belongs to which side is **detected from the ent_ids content**,
not assumed, and the join is checked — a key-format mismatch would otherwise leave every
entity attribute-less and produce a meaningless run in silence.

In [4]:
import re
from collections import defaultdict

NT_RE = re.compile(r'^<([^>]+)>\s+<([^>]+)>\s+(.*?)\s*\.\s*$')
VALUE_RE = re.compile(r'^"(.*)"(?:\^\^<[^>]+>|@[\w-]+)?$', re.DOTALL)
KG_OF = {'FBDB15K': ('FB15K', 'DB15K'), 'FBYG15K': ('FB15K', 'YAGO15K')}

def short(u):     return u.rsplit('/', 1)[-1].rsplit('#', 1)[-1]
def norm_key(k):  return k.strip().lstrip('<').rstrip('>')

def read_ent_ids(path):
    out = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line:
                eid, key = line.split('\t')
                out[norm_key(key)] = int(eid)
    return out

def detect_kg(uri2id):
    s = list(uri2id)[:200]
    if sum(k.startswith('/m/') for k in s) > len(s) // 2:      return 'FB15K'
    if sum('dbpedia.org' in k for k in s) > len(s) // 2:       return 'DB15K'
    if sum('yago' in k.lower() for k in s) > len(s) // 2:      return 'YAGO15K'
    raise SystemExit(f'cannot tell which KG this is; sample: {s[:3]}')

def parse_fb15k(path, uri2id, cap):
    per, kept, skip = defaultdict(list), 0, 0
    for line in open(path, encoding='utf-8'):
        p = line.rstrip('\n').split('\t')
        if len(p) != 3:
            skip += 1; continue
        eid = uri2id.get(norm_key(p[0]))
        if eid is None: continue
        v = p[2].strip().strip('"')
        if not v or len(per[eid]) >= cap: continue
        per[eid].append(f'({short(norm_key(p[1]))}, {v})'); kept += 1
    return per, kept, skip

def parse_ntriples(path, uri2id, cap):
    per, kept, skip = defaultdict(list), 0, 0
    for line in open(path, encoding='utf-8'):
        m = NT_RE.match(line.rstrip('\n'))
        if not m:
            skip += 1; continue
        subj, pred, raw = m.groups()
        eid = uri2id.get(norm_key(subj))
        if eid is None: continue
        vm = VALUE_RE.match(raw.strip())
        v = (vm.group(1) if vm else raw.strip()).replace('\\"', '"').replace('\n', ' ').strip()
        if not v or len(per[eid]) >= cap: continue
        per[eid].append(f'({short(pred)}, {v})'); kept += 1
    return per, kept, skip

ENT_NUM = {}
for d in DATASETS:
    dd = ds_dir(d)
    sides = [('attribute1.txt', read_ent_ids(f'{dd}/ent_ids_1')),
             ('attribute2.txt', read_ent_ids(f'{dd}/ent_ids_2'))]
    seen = set()
    for _, u2i in sides:
        assert not (seen & set(u2i.values())), f'{d}: entity ids collide across the two KGs'
        seen |= set(u2i.values())
    ENT_NUM[d] = len(seen)

    if os.path.exists(f'{dd}/attribute1.txt') and os.path.exists(f'{dd}/ent2id'):
        print(f'{d}: attribute files already present ({ENT_NUM[d]} entities)')
        continue

    expected = set(KG_OF[d])
    for out_name, u2i in sides:
        kg = detect_kg(u2i)
        assert kg in expected, f'{out_name} looks like {kg}, expected one of {expected}'
        expected.discard(kg)
        path = f'{MMKB}/{kg}/{kg}_NumericalTriples.txt'
        assert os.path.exists(path), f'missing {path}'
        per, kept, skip = (parse_fb15k if kg == 'FB15K' else parse_ntriples)(
            path, u2i, MAX_ATTR_PER_ENTITY)

        cov = len(per) / len(u2i)
        if cov < 0.30:
            raw = [next(open(path, encoding='utf-8')).split('\t')[0].strip()]
            raise SystemExit(
                f'{out_name}: only {cov:.1%} of {kg} entities matched an attribute.\n'
                f'  ent_ids keys : {list(u2i)[:3]}\n  mmkb subjects: {raw}\n'
                f'The key formats disagree -- fix norm_key() before continuing.')

        n_empty = 0
        with open(f'{dd}/{out_name}', 'w', encoding='utf-8') as f:
            for eid in sorted(u2i.values()):
                items = per.get(eid, [])
                n_empty += not items
                f.write(f'{eid}\t{items!r}\n')
        print(f'{d}/{out_name} [{kg}]: {len(u2i)} entities, {len(per)} with attrs ({cov:.1%}), '
              f'{n_empty} empty, {kept} pairs, {skip} unparsed')

    with open(f'{dd}/ent2id', 'w', encoding='utf-8') as f:
        for n in ('ent_ids_1', 'ent_ids_2'):
            for line in open(f'{dd}/{n}', encoding='utf-8'):
                if line.strip():
                    f.write(line if line.endswith('\n') else line + '\n')
    print(f'{d}/ent2id: {ENT_NUM[d]} lines')

FBDB15K/attribute1.txt [FB15K]: 14951 entities, 12493 with attrs (83.6%), 2458 empty, 29395 pairs, 0 unparsed
FBDB15K/attribute2.txt [DB15K]: 12842 entities, 10985 with attrs (85.5%), 1857 empty, 48069 pairs, 36 unparsed
FBDB15K/ent2id: 27793 lines


In [5]:
# --- verify before spending GPU time ---
import pickle

for d in DATASETS:
    dd = ds_dir(d)
    n_attr = sum(1 for n in ('attribute1.txt', 'attribute2.txt')
                   for _ in open(f'{dd}/{n}', encoding='utf-8'))
    n_e2i = sum(1 for _ in open(f'{dd}/ent2id', encoding='utf-8'))
    with open(f'{PKL_DIR}/{img_pkl(d)}', 'rb') as f:
        img = pickle.load(f)
    ok = 0
    for n in ('attribute1.txt', 'attribute2.txt'):
        for line in open(f'{dd}/{n}', encoding='utf-8'):
            v = eval(line.rstrip('\n').split('\t')[1])     # exactly what load_attr does
            assert isinstance(v, list) and all(isinstance(x, str) for x in v)
            ok += 1
    print(f'{d}: ENT_NUM={ENT_NUM[d]}  ent2id={n_e2i}  attr_lines={n_attr} ({ok} eval-clean)  '
          f'images={len(img)} ({100*len(img)/ENT_NUM[d]:.1f}%, dim={len(next(iter(img.values())))})')
    assert n_e2i == ENT_NUM[d] == n_attr
print('\nSTAGE 1 OK')

FBDB15K: ENT_NUM=27793  ent2id=27793  attr_lines=27793 (27793 eval-clean)  images=26281 (94.6%, dim=4096)

STAGE 1 OK


## 4 · Attribute features

Two sets, because Table 3 needs both:

* **LLM** (`att_sum_*_noname_<ds>.pkl`) — LLM2Vec Llama-3-8B, 4096-d. This is LGEA.
* **bag-of-attributes** (`att_bag_*_<ds>.pkl`) — top-1000 binary indicator over
  `training_attrs_*`, which is what MEAformer used before LGEA replaced it. This is the
  `w/o llm` row, and it needs no GPU.

Writing the bag into the *same pkl format* means `w/o llm` is a file swap, with no patch
to `src/data.py`.

In [6]:
# --- w/o llm: bag-of-attributes, top 1000 by frequency (MEAformer's original encoding) ---
import numpy as np
from collections import Counter

for d in DATASETS:
    if os.path.exists(bag_path('left', d)) and not FORCE_EMBED:
        print(f'{d}: bag features already built'); continue
    dd = ds_dir(d)
    cnt = Counter()
    rows = {1: {}, 2: {}}
    for side in (1, 2):
        u2i = read_ent_ids(f'{dd}/ent_ids_{side}')
        for line in open(f'{dd}/training_attrs_{side}', encoding='utf-8'):
            th = line.rstrip('\n').split('\t')
            eid = u2i.get(norm_key(th[0]))
            if eid is None: continue
            attrs = th[1:]
            cnt.update(attrs)
            rows[side][eid] = attrs
    attr2id = {a: i for i, (a, _) in enumerate(cnt.most_common(1000))}
    for side, name in ((1, 'left'), (2, 'right')):
        out = {}
        for eid, attrs in rows[side].items():
            v = np.zeros(1000, dtype=np.float32)
            for a in attrs:
                if a in attr2id: v[attr2id[a]] = 1.0
            if v.any(): out[str(eid)] = v
        with open(bag_path(name, d), 'wb') as f:
            pickle.dump(out, f)
        print(f'{d} {name}: bag-of-attributes {len(out)} entities x 1000')

FBDB15K left: bag-of-attributes 12493 entities x 1000
FBDB15K right: bag-of-attributes 10985 entities x 1000


In [7]:
# Prompt is Table 11 of the paper, verbatim.
PROMPT = '''
You are an expert who can provide concise explanations based on entity information.
I will give you the properties of attributes and values of an entity in the form of (predicate object).
Using these information, please provide a short description of the entity.

- The explanation should be no longer than 100 words.
- Focus on summarizing the entity based on the given information.
- Do not include unnecessary details or explanations beyond the entity description.
- Do not include entity name.

Example:

Entity Information: (occupation Mathematician), (notable work Principia Mathematica)
Explanation: The entity was a prominent mathematician recognized for authoring Principia
Mathematica, a foundational work in the history of science and mathematics.

Now, please summarize the following entity information and return an description in English:
'''

def build_pairs(dataset, which):
    ids, pairs, skipped = [], [], 0
    with open(f'{ds_dir(dataset)}/attribute{which}.txt', encoding='utf-8') as f:
        for line in f:
            eid, info = line.rstrip('\n').split('\t')
            items = eval(info)
            if not items:
                skipped += 1; continue
            ids.append(eid)
            pairs.append([PROMPT, ';'.join(items)])
    print(f'{dataset}/attribute{which}.txt: {len(ids)} to embed, {skipped} zero-filled')
    return ids, pairs

if EMB_INPUT and not FORCE_EMBED:
    for d in DATASETS:
        for side, which in (('left', 1), ('right', 2)):
            want = sum(1 for line in open(f'{ds_dir(d)}/attribute{which}.txt',
                                          encoding='utf-8')
                       if eval(line.rstrip('\n').split('\t')[1]))
            src = f'{EMB_INPUT}/att_sum_{side}_noname_{d}.pkl'
            assert os.path.exists(src), f'{src} not found'
            with open(src, 'rb') as f:
                loaded = pickle.load(f)
            assert len(loaded) == want, (
                f'{src} holds {len(loaded)} entities but {d} needs {want} '
                f'-- this file is from a different dataset')
            shutil.copy2(src, emb_path(side, d))
            print(f'{d} {side}: reused {len(loaded)} embeddings')
        mark(f'embed_{d}')

pending = [d for d in DATASETS if FORCE_EMBED or not done(f'embed_{d}')]
print('\ndatasets needing LLM embedding:', pending or 'none')


datasets needing LLM embedding: ['FBDB15K']


In [8]:
import torch, time, gc

if not pending:
    print('all embeddings present, skipping the 8B model entirely')
else:
    from transformers import AutoTokenizer, AutoModel, AutoConfig
    from peft import PeftModel
    from peft.tuners.tuners_utils import BaseTunerLayer
    from llm2vec import LLM2Vec
    from tqdm.auto import trange

    if BASE_MODEL.startswith('meta-llama'):
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

    tokenizer = AutoTokenizer.from_pretrained(L2V_MNTP, revision=CODE_REV)
    config = AutoConfig.from_pretrained(L2V_MNTP, trust_remote_code=True, revision=CODE_REV)

    # device_map='auto' splits a decoder layer here: transformers lists
    # _no_split_modules = ["LlamaDecoderLayer"] but LLM2Vec builds
    # ModifiedLlamaDecoderLayer, and accelerate matches by class NAME. A hand-written map
    # keeps every layer whole.
    n_layers = config.num_hidden_layers
    ls = n_layers // 2 - 1
    device_map = {'embed_tokens': 0}
    device_map.update({f'layers.{i}': (0 if i < ls else 1) for i in range(n_layers)})
    device_map['norm'] = 1

    model = AutoModel.from_pretrained(BASE_MODEL, trust_remote_code=True, config=config,
                                      code_revision=CODE_REV, torch_dtype=torch.float16,
                                      device_map=device_map, low_cpu_mem_usage=True)
    for i, layer in enumerate(model.layers):
        assert len({str(p.device) for p in layer.parameters()}) == 1, f'layer {i} split'
    print(f'{n_layers} layers intact')

    model = PeftModel.from_pretrained(model, L2V_MNTP, revision=CODE_REV)
    model = model.merge_and_unload()
    model = PeftModel.from_pretrained(model, L2V_SUP)

    # L2V_SUP is loaded after merge_and_unload() stripped accelerate's hooks, so peft sees
    # no hf_device_map and drops every adapter tensor on cuda:0. Put them back.
    def adapter_tensors(m):
        for attr in list(m.adapter_layer_names) + list(m.other_param_names):
            dd_ = getattr(m, attr, None)
            if hasattr(dd_, 'items'):
                for k, sub in dd_.items():
                    yield attr, k, sub

    n_ad = 0
    for m in model.modules():
        if isinstance(m, BaseTunerLayer):
            n_ad += 1
            for k in {k for _, k, _ in adapter_tensors(m)}:
                m._move_adapter_to_device_of_base_layer(k)
    bad = []
    for name, m in model.named_modules():
        if not isinstance(m, BaseTunerLayer): continue
        base = getattr(m.get_base_layer(), 'weight', None)
        if base is None: continue
        for attr, k, sub in adapter_tensors(m):
            for p in getattr(sub, 'parameters', lambda: [])():
                if p.device != base.device:
                    bad.append(f'{name}.{attr}.{k}')
    print(f'realigned {n_ad} adapter layers, {len(bad)} still mismatched')
    assert not bad, bad[:5]

    model.eval()
    l2v = LLM2Vec(model, tokenizer, pooling_mode='mean', max_length=MAX_LENGTH)

    def encode_sharded(pairs, desc):
        '''llm2vec.encode() forks a process per GPU and pickles the model (llm2vec.py:357),
        which assumes data parallel. We need pipeline parallel, so run its single-process
        branch by hand -- same sort/restore order, no self.to() call.'''
        texts = [l2v._convert_to_str(i_, t) for i_, t in pairs]
        l2v.eval()
        order = np.argsort([-l2v._text_length(t) for t in texts])
        ordered = [texts[i] for i in order]
        dev = next(l2v.model.parameters()).device
        chunks = []
        for i in trange(0, len(ordered), BATCH_SIZE, desc=desc):
            feats = l2v.tokenize([l2v.prepare_for_tokenization(t)
                                  for t in ordered[i:i + BATCH_SIZE]])
            feats = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in feats.items()}
            with torch.no_grad():
                chunks.append(l2v.forward(feats).detach().cpu())
        return torch.cat(chunks, 0)[np.argsort(order)].to(torch.float32).numpy()

    probe = encode_sharded(build_pairs(pending[0], 1)[1][:2], 'smoke')
    assert probe.shape == (2, 4096) and np.isfinite(probe).all() and np.linalg.norm(probe[0]) > 0
    print(f'smoke test ok: {probe.shape}')

    for d in pending:
        t0 = time.time()
        for side, which in (('left', 1), ('right', 2)):
            ids, pairs = build_pairs(d, which)
            emb = encode_sharded(pairs, f'{d}:{side}')
            with open(emb_path(side, d), 'wb') as f:
                pickle.dump({eid: emb[i] for i, eid in enumerate(ids)}, f)
            print(f'{d} {side}: {emb.shape}')
        mark(f'embed_{d}')
        print(f'{d} embedded in {(time.time()-t0)/60:.1f} min\n')

    del l2v, model, tokenizer, config
    gc.collect(); torch.cuda.empty_cache()

print('GPU free:', [f'{torch.cuda.mem_get_info(i)[0]/1e9:.1f}G'
                    for i in range(torch.cuda.device_count())])

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

modeling_llama_encoder.py: 0.00B [00:00, ?B/s]

...

smoke test ok: (2, 4096)
FBDB15K/attribute1.txt: 12493 to embed, 2458 zero-filled


FBDB15K:left:   0%|          | 0/781 [00:00<?, ?it/s]

FBDB15K left: (12493, 4096)
FBDB15K/attribute2.txt: 10985 to embed, 1857 zero-filled


FBDB15K:right:   0%|          | 0/687 [00:00<?, ?it/s]

FBDB15K right: (10985, 4096)
FBDB15K embedded in 66.0 min

GPU free: ['15.5G', '15.0G']


## 5 · Patch `model/layers.py`

`SpecialSpmmFunction.backward` builds the full N×N dense product `grad_output @ b.t()` and
then gathers only E entries from it. At N≈28k that is a 2.9 GiB tensor, allocated twice per
attention head; it is what OOMs a 15GB T4 on DBP15K (N=38,960 → 5.66 GiB). Since
`grad_a_dense[i,j] == sum_k grad_output[i,k] * b[j,k]`, the E entries actually used are
computed edge-wise instead — **identical numbers**, O(E·d) rather than O(N²).

In [9]:
LAYERS = f'{CODE}/model/layers.py'
ORIGINAL = """            grad_a_dense = grad_output.matmul(b.t())
            edge_idx = a._indices()[0, :] * ctx.N + a._indices()[1, :]
            grad_values = grad_a_dense.view(-1)[edge_idx]"""

PATCHED = """            # PATCHED (memory only -- numerically identical).
            # The original built the full N x N dense product grad_output @ b.t() and
            # then gathered E entries from it. grad_a_dense[i, j] == sum_k
            # grad_output[i, k] * b[j, k], so compute only the E entries we need,
            # chunked, accumulating in fp32.
            _row = a._indices()[0, :]
            _col = a._indices()[1, :]
            _E = _row.numel()
            _step = max(1, int(2e7) // max(1, b.size(1)))
            grad_values = torch.empty(_E, device=grad_output.device, dtype=grad_output.dtype)
            for _s in range(0, _E, _step):
                _e = min(_s + _step, _E)
                grad_values[_s:_e] = (
                    grad_output.index_select(0, _row[_s:_e]).float()
                    * b.index_select(0, _col[_s:_e]).float()
                ).sum(dim=1).to(grad_output.dtype)"""

source = open(LAYERS, encoding='utf-8').read()
if 'PATCHED (memory only' in source:
    print('layers.py already patched')
else:
    assert source.count(ORIGINAL) == 1, 'unexpected backward() body -- upstream changed'
    open(LAYERS, 'w', encoding='utf-8').write(source.replace(ORIGINAL, PATCHED))
    print('patched SpecialSpmmFunction.backward')

recheck = open(LAYERS, encoding='utf-8').read()
assert 'grad_output.matmul(b.t())' not in recheck
import ast; ast.parse(recheck)
print('layers.py parses clean, no N x N allocation remains')

patched SpecialSpmmFunction.backward
layers.py parses clean, no N x N allocation remains


## 6 · Run the matrix

| row | how |
|---|---|
| `LGEA` | baseline flags (= the paper's `w/o mask`, since we have no mask) |
| `w/o img` | `--w_img` — the flag is `store_false`, so passing it turns the modality **off** |
| `w/o rel` | `--w_rel` |
| `w/o structure` | `--w_gcn` |
| `w/o att` | `--w_attr` |
| `w/o llm` | swap in the bag-of-attributes pkl instead of the LLM one |
| `w/o it` | drop `--il` (250 epochs instead of 1000) |

In [10]:
ABLATION_FLAGS = {
    'LGEA': [], 'w/o llm': [], 'w/o it': [],
    'w/o img': ['--w_img'], 'w/o rel': ['--w_rel'],
    'w/o structure': ['--w_gcn'], 'w/o att': ['--w_attr'],
}
assert all(r in ABLATION_FLAGS for r in RUNS), [r for r in RUNS if r not in ABLATION_FLAGS]

# expandable_segments: an earlier run held 10.59 GB with only 4.65 GB live.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

for d in DATASETS:
    dd = ds_dir(d)
    for run in RUNS:
        tag = f'train_{d}_{slug(run)}'
        if done(tag) and not FORCE_TRAIN:
            print(f'{d} / {run}: already done, skipping'); continue

        # attribute features: LLM everywhere except the w/o llm row
        src = bag_path if run == 'w/o llm' else emb_path
        for side in ('left', 'right'):
            shutil.copy2(src(side, d), f'{dd}/att_sum_{side}_noname.pkl')
        with open(f'{dd}/att_sum_left_noname.pkl', 'rb') as f:
            dim = len(next(iter(pickle.load(f).values())))
        print(f'\n===== {d} / {run} =====  attr features: '
              f'{"bag-of-attributes" if run == "w/o llm" else "LLM2Vec"} ({dim}-d)')

        il = USE_IL and run != 'w/o it'
        cmd = ['python', 'main.py',
               '--gpu', '0', '--eval_epoch', '1', '--only_test', '0',
               '--model_name', 'NewEA', '--data_choice', d,
               '--data_split', 'norm', '--data_rate', DATA_RATE,
               '--epoch', '1000' if il else '500',   # enable_sota clamps non-IL to 250
               '--lr', '5e-4', '--hidden_units', '300,300,300',
               '--batch_size', '3500',
               '--csls', '--csls_k', '3', '--random_seed', SEED,
               '--exp_name', f'LGEA_{slug(run)}', '--exp_id', f'v1_{d}_{DATA_RATE}',
               '--workers', '2', '--dist', '0', '--accumulation_steps', '1',
               '--scheduler', 'cos',
               '--attr_dim', '300', '--img_dim', '300',
               '--name_dim', '300', '--char_dim', '300', '--hidden_size', '300',
               '--tau', '0.1', '--structure_encoder', 'gat',
               '--num_attention_heads', '1', '--num_hidden_layers', '1',
               '--use_surface', '0', '--use_intermediate', '1', '--enable_sota']
        if il:
            cmd += ['--il', '--il_start', '500', '--semi_learn_step', '5']
        if SAVE_MODEL:
            cmd += ['--save_model', '1']
        cmd += ABLATION_FLAGS[run]
        print(' '.join(cmd) + '\n')

        proc = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
        proc.wait()
        assert proc.returncode == 0, f'{d}/{run}: training exited {proc.returncode}'
        mark(tag)


===== FBDB15K / LGEA =====  attr features: LLM2Vec (4096-d)
python main.py --gpu 0 --eval_epoch 1 --only_test 0 --model_name NewEA --data_choice FBDB15K --data_split norm --data_rate 0.2 --epoch 1000 --lr 5e-4 --hidden_units 300,300,300 --batch_size 3500 --csls --csls_k 3 --random_seed 42 --exp_name LGEA_LGEA --exp_id v1_FBDB15K_0.2 --workers 2 --dist 0 --accumulation_steps 1 --scheduler cos --attr_dim 300 --img_dim 300 --name_dim 300 --char_dim 300 --hidden_size 300 --tau 0.1 --structure_encoder gat --num_attention_heads 1 --num_hidden_layers 1 --use_surface 0 --use_intermediate 1 --enable_sota --il --il_start 500 --semi_learn_step 5 --save_model 1

INFO - 10/04/26 14:01:19 - 0:00:00 - ============ Initialized logger ============
INFO - 10/04/26 14:01:19 - 0:00:00 - The experiment will be stored in /kaggle/working/lgea/data/mmkg/dump/1004-LGEA_LGEA/NewEA_FBDB15K_0.2_v1_FBDB15K_0.2
                                     
INFO - 10/04/26 14:01:19 - 0:00:00 - Running command: python main.

## 7 · Results

The log never prints the string `hits@1`; it writes
`acc of top [1, 10, 50] = [0.9578 0.9974 1.    ], mr = ..., mrr = 0.975`, so the parser
reads that form from the final block after `--------- Test result ---------`.

For the `LGEA` row the honest comparison is the paper's **`w/o mask`**, shown as `paper*`.

In [11]:
import glob

TABLE2 = {'FBDB15K': (.801, .910, .842), 'FBYG15K': (.850, .942, .883)}
TABLE3 = {  # Hits@1 / Hits@10 / MRR, from Table 3
    'FBDB15K': {'LGEA': (.801, .910, .842), 'w/o img': (.740, .890, .791),
                'w/o rel': (.737, .890, .791), 'w/o structure': (.440, .615, .502),
                'w/o att': (.655, .840, .720), 'w/o llm': (.682, .855, .745),
                'w/o mask': (.714, .868, .772), 'w/o it': (.684, .868, .749)},
    'FBYG15K': {'LGEA': (.850, .942, .883), 'w/o img': (.767, .896, .811),
                'w/o rel': (.813, .917, .851), 'w/o structure': (.678, .844, .737),
                'w/o att': (.563, .755, .628), 'w/o llm': (.558, .756, .626),
                'w/o mask': (.818, .917, .854), 'w/o it': (.785, .914, .832)},
}
ROW = re.compile(r'(l2r|r2l): acc of top \[1, 10, 50\] = \[([\d.\s]+)\].*?mrr = ([\d.]+)')

for d in DATASETS:
    print(f'\n===== {d} =====')
    print(f'{"row":15s} {"Hits@1":>15s} {"Hits@10":>15s} {"MRR":>15s}   note')
    print(f'{"":15s} {"paper / ours":>15s} {"paper / ours":>15s} {"paper / ours":>15s}')
    for run in RUNS:
        logs = sorted(glob.glob(f'{DATA}/dump/*LGEA_{slug(run)}*/*{d}*/train.log'),
                      key=os.path.getmtime)
        if not logs:
            print(f'{run:15s} no log'); continue
        text = open(logs[-1], errors='ignore').read()
        if 'Test result' not in text:
            print(f'{run:15s} unfinished run'); continue
        rows = ROW.findall(text[text.rindex('Test result'):])
        if not rows:
            print(f'{run:15s} no parsable test block'); continue
        _, accs, mrr = rows[0]                       # l2r, as the paper reports
        a = [float(x) for x in accs.split()]
        key, note = (('w/o mask', '= paper w/o mask') if run == 'LGEA' else (run, ''))
        p = TABLE3[d][key]
        print(f'{run:15s} {p[0]:6.3f} / {a[0]:6.4f} {p[1]:6.3f} / {a[1]:6.4f} '
              f'{p[2]:6.3f} / {float(mrr):6.3f}   {note}')
    print(f'{"(Table 2 full)":15s} {TABLE2[d][0]:6.3f}          {TABLE2[d][1]:6.3f}'
          f'          {TABLE2[d][2]:6.3f}          needs the visual mask')

print(f'\nConfig: USE_IL={USE_IL}, seeds={DATA_RATE}, no visual mask.')
print('Ablation rows other than LGEA include the mask in the paper but not here, so read '
      'their SHAPE (which modality hurts most), not the absolute gap.')


===== FBDB15K =====
row                      Hits@1         Hits@10             MRR   note
                   paper / ours    paper / ours    paper / ours
LGEA             0.714 / 0.7322  0.868 / 0.8798  0.772 /  0.786   = paper w/o mask
w/o img          0.740 / 0.7147  0.890 / 0.8640  0.791 /  0.767   
w/o rel          0.737 / 0.6688  0.890 / 0.8455  0.791 /  0.732   
w/o structure    0.440 / 0.4030  0.615 / 0.5764  0.502 /  0.466   
w/o att          0.655 / 0.5402  0.840 / 0.7646  0.720 /  0.621   
w/o llm          0.682 / 0.5570  0.855 / 0.7834  0.745 /  0.637   
w/o it           0.684 / 0.5987  0.868 / 0.8192  0.749 /  0.676   
(Table 2 full)   0.801           0.910           0.842          needs the visual mask

Config: USE_IL=True, seeds=0.2, no visual mask.
Ablation rows other than LGEA include the mask in the paper but not here, so read their SHAPE (which modality hurts most), not the absolute gap.
